### Reddit Praw

In [4]:
#pip install praw

In [16]:
#pip install python-dotenv

In [23]:
import pandas as pd
import praw
from datetime import datetime as dt
from dotenv import load_dotenv
from os import getenv

# load .env file
load_dotenv()

True

The data came from a web API, more precisely the Reddit API. When you retrieved the data via the API, it was most likely in JSON format, which is normal for most online APIs. This information was then transformed into a pandas DataFrame for Python analysis and manipulation.

#### # Function collect_reddit: Collects comment from Reddit, it receives the subreddit and the search_query as Paramter, and returns a dataframe

In [35]:
def collect_reddit(subreddit, search_query):
    try:
        reddit = praw.Reddit(
            client_id=getenv('APP_ID'),
            client_secret=getenv('APP_SECRET'),
            user_agent=f"{getenv('APP_NAME')} u/{getenv('REDDIT_USERNAME')}",)

        subreddit = reddit.subreddit(subreddit)
        results = subreddit.search(search_query, limit=500)

        titles = []
        texts = []

        for submission in results:
            titles.append(submission.title)
            texts.append(submission.selftext)

        if not titles:
            print("No data retrieved. Check subreddit name, search query, or API credentials.")
        else:
            print(f"Retrieved {len(titles)} posts.")
        return pd.DataFrame({'title': titles, 'text': texts})
    except Exception as e:
        print(f"An error occurred: {e}")

# The function
transp_data = collect_reddit('Dublin', 'Dublin public transport')

# Check if the DataFrame is empty
if transp_data.empty:
    print("The DataFrame is empty.")
else:
    print(transp_data.head())

Retrieved 236 posts.
                                               title  \
0  Dublin public transport is light years ahead o...   
1               Public transport in Dublin is a joke   
2  Is there a european capital with worse public ...   
3  Why Public transportation in Dublin is so shit...   
4  Dublin ranked worst for public transport in Eu...   

                                                text  
0  A lot of negativity about the fair city these ...  
1  Currently in the city center. No busses in my ...  
2  I have been in maybe 10 European capitals and ...  
3  The buses are ALWAYS late, the time tables are...  
4                                                     


# Convert DataFrame to JSON
json_data = transp_data.to_json(orient='records', lines=True)

# To save the JSON data to a file
with open('transp_data.json', 'w') as file:
    file.write(json_data)
    
print(transp_data.head())

In [36]:
# Import libraries
import pandas as pd
import re
# Convert DataFrame to CSV
transp_data.to_csv('transp_data.csv', index=False)

# Confirm the operation
print("Data saved to 'transp_data.csv'")

Data saved to 'transp_data.csv'


In [37]:
transp_data.head()

,title,text
0,Dublin public transport is light years ahead o...,A lot of negativity about the fair city these ...
1,Public transport in Dublin is a joke,Currently in the city center. No busses in my ...
2,Is there a european capital with worse public ...,I have been in maybe 10 European capitals and ...
3,Why Public transportation in Dublin is so shit...,"The buses are ALWAYS late, the time tables are..."
4,Dublin ranked worst for public transport in Eu...,


In [38]:
transp_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236 entries, 0 to 235
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   236 non-null    object
 1   text    236 non-null    object
dtypes: object(2)
memory usage: 3.8+ KB


In [40]:
transp_data.shape

(236, 2)

In [42]:
# Remove duplicates
transp_data.drop_duplicates(inplace=True)

# Data cleaning function
def clean_text(text):
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'@\w+', '', text)  # Remove mentions
    text = re.sub(r'#\w+', '', text)  # Remove hashtags
    text = re.sub(r'\d+', '', text)  # Remove numbers
    text = re.sub(r'\W+', ' ', text)  # Remove non-alphanumeric characters
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with a single space
    text = text.strip()  # Remove leading and trailing spaces
    text = text.lower()  # Convert to lowercase
    return text

# Clean the text data
transp_data['cleaned_text'] = transp_data['text'].apply(clean_text)

In [46]:
transp_data.head()

,title,text,cleaned_text,sentiment
0,Dublin public transport is light years ahead o...,A lot of negativity about the fair city these ...,a lot of negativity about the fair city these ...,0.550000
1,Public transport in Dublin is a joke,Currently in the city center. No busses in my ...,currently in the city center no busses in my d...,0.099074
2,Is there a european capital with worse public ...,I have been in maybe 10 European capitals and ...,i have been in maybe european capitals and the...,0.012500
3,Why Public transportation in Dublin is so shit...,"The buses are ALWAYS late, the time tables are...",the buses are always late the time tables are ...,-0.650000
4,Dublin ranked worst for public transport in Eu...,,,0.000000


In [ ]:
transp_data()

#### Sentiment Analysis

In [44]:
from textblob import TextBlob

# Function to get sentiment
def get_sentiment(text):
    return TextBlob(text).sentiment.polarity

# Apply the function
transp_data['sentiment'] = transp_data['cleaned_text'].apply(get_sentiment)

#### Topic Modeling

In [45]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Initialize CountVectorizer
vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words='english')
doc_term_matrix = vectorizer.fit_transform(transp_data['cleaned_text'])

# LDA model
lda = LatentDirichletAllocation(n_components=5, random_state=0)
lda.fit(doc_term_matrix)

LatentDirichletAllocation(n_components=5, random_state=0)

In [ ]:
#### Visualization
# For topic visualization, extract words for each topic from LDA model and create bar plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting sentiment distribution
sns.histplot(transp_data['sentiment'], kde=False)
plt.title('Sentiment Distribution')
plt.xlabel('Sentiment Score')
plt.ylabel('Frequency')
plt.show()